# 2. Analytics Rules and Detection

**Analytics rules** are the detection engine of a SIEM: scheduled queries that run against your log data and fire *alerts* when patterns match.

### What you'll learn
- The six types of Sentinel analytics rules
- The bad → best progression for writing a rule (threshold, window, aggregation, entity mapping)
- How to tune a noisy rule (reduce false positives)
- MITRE ATT&CK coverage as a SOC maturity metric

## Types of Sentinel analytics rules

| Type | How it works | Latency | Use case |
|------|-------------|---------|----------|
| **Scheduled** | KQL runs on a schedule (e.g., every 5 min) | Minutes | Most detections |
| **NRT (Near Real-Time)** | KQL runs ~every minute | ~1 minute | Time-critical threats |
| **Microsoft Security** | Imports alerts from other Defender products | Seconds | XDR correlation |
| **Threat Intelligence** | Matches IOCs against log data | Minutes | Known-bad IPs, domains |
| **Anomaly** | ML-based baseline deviation | Varies | Unusual behavior |
| **Fusion** | Multi-stage attack correlation (ML) | Minutes | Advanced attacks |

## 0. Setup — pick the lab kernel

This lab has its own `uv`-managed virtual environment. Before running any code cell:

1. From `security-certs/sc-200/01-build-a-siem/` run once in a terminal:
   ```bash
   uv sync
   docker compose up -d
   ```
2. In VS Code, click the kernel picker (top-right of this notebook) and choose **`.venv (Python 3.xx)`** from this folder.
3. If the kernel does not appear, reload the window: `Cmd+Shift+P` → `Reload Window`.

The log-generator container has already seeded the SIEM with normal traffic **and** four attack patterns (brute force, lateral movement, exfiltration, phishing). Every cell below talks to `http://localhost:8000`.

In [ ]:
import httpx, json

SIEM = 'http://localhost:8000'

print('=== Current Analytics Rules ===')
rules = httpx.get(f'{SIEM}/rules').json()
for r in rules:
    sev = {'Critical':'🟣','High':'🔴','Medium':'🟡','Low':'🟢'}.get(r['severity'], '⬜')
    print(f'  {sev} [{r["id"]}] {r["name"]}')
    print(f'     Table: {r["query_table"]}  |  Tactic: {r["tactic"] or "-"}  |  Window: {r["window_minutes"]}min  |  Threshold: >={r["threshold"]}')
    if r['query_filter']: print(f'     Filter: {r["query_filter"]}')
    if r['aggregate_by']: print(f'     Group by: {r["aggregate_by"]}')
    print()

## 2.1 Bad rule → Best rule progression

New SOCs often start with **naive** rules that alert on any single failed sign-in. The result: hundreds of alerts, alert fatigue, real attacks get ignored. Let's build the same detection three ways and compare.

> Re-running these cells is safe — our mini-SIEM upserts rules by name.

In [ ]:
# BAD: any single failure triggers an alert. Pure noise.
httpx.post(f'{SIEM}/rules', json={
    'name': 'Demo - failed sign-in (bad)',
    'severity': 'Low',
    'tactic': 'CredentialAccess',
    'query_table': 'SigninLogs',
    'query_filter': {'ResultType': 'Failure'},
    'threshold': 1,
    'window_minutes': 1440,
    'description': 'Fires on every failed sign-in. Will drown the SOC in noise.',
})

# BETTER: add a threshold so isolated typos dont alert
httpx.post(f'{SIEM}/rules', json={
    'name': 'Demo - failed sign-in (better)',
    'severity': 'Medium',
    'tactic': 'CredentialAccess',
    'query_table': 'SigninLogs',
    'query_filter': {'ResultType': 'Failure'},
    'threshold': 10,
    'window_minutes': 60,
    'description': '10+ failures in 1h. Still alerts once per org - no attribution.',
})

# BEST: aggregate per user, high severity, mapped to ATT&CK
httpx.post(f'{SIEM}/rules', json={
    'name': 'Demo - failed sign-in (best)',
    'severity': 'High',
    'tactic': 'CredentialAccess',
    'query_table': 'SigninLogs',
    'query_filter': {'ResultType': 'Failure'},
    'aggregate_by': 'UserPrincipalName',
    'threshold': 5,
    'window_minutes': 60,
    'description': '5+ failures per user in 1h. Entity = user -> feeds incident correlation.',
})
print('Created 3 demo rules: bad / better / best.')

In [ ]:
# Evaluate the three demo rules and compare
r = httpx.post(f'{SIEM}/rules/evaluate').json()
demo_alerts = [a for a in r['alerts_created'] if a['rule'].startswith('Demo - ')]
print(f'Demo alerts fired: {len(demo_alerts)}\n')
for a in demo_alerts:
    print(f'  🚨 {a["rule"]}: group={a.get("group","-"):<25} count={a["count"]}')
print('\nKey takeaway:')
print('  bad    -> lumps everything into one huge alert (low signal, no entity)')
print('  better -> fewer alerts but still just one aggregate count')
print('  best   -> one alert per *affected user* = SOC can act on each entity')

## 2.2 Tuning a noisy rule — real-world false-positive reduction

A detection that constantly alerts on normal behaviour is worse than no detection at all. Watch what happens when we create an over-broad rule, then tune it:

In [ ]:
# Noisy rule: any new process on the DB server. Looks strict, but normal admins run things too.
httpx.post(f'{SIEM}/rules', json={
    'name': 'DB-server new process (over-broad)',
    'severity': 'High',
    'tactic': 'Execution',
    'query_table': 'DeviceEvents',
    'query_filter': {'DeviceName': 'vm-db-01', 'ActionType': 'ProcessCreated'},
    'aggregate_by': 'FileName',
    'threshold': 1,
    'window_minutes': 120,
})
r = httpx.post(f'{SIEM}/rules/evaluate').json()
db_alerts = [a for a in r['alerts_created'] if a['rule'] == 'DB-server new process (over-broad)']
print(f'Over-broad rule fired {len(db_alerts)} alerts. Sample groups:')
for a in db_alerts[:8]:
    verdict = 'FP' if a['group'] in ('chrome.exe','code','python3','outlook.exe') else 'TP'
    print(f'  [{verdict}] {a["group"]:<15} ({a["count"]} events)')
print('\nchrome/code/python3 are developer noise on that host, not an attack.')

In [ ]:
# Tuned rule: only alert on known attacker tools. Massive false-positive drop.
# In production you'd also maintain an *allowlist* watchlist for known-good admin binaries.
httpx.post(f'{SIEM}/rules', json={
    'name': 'DB-server new process (tuned)',
    'severity': 'High',
    'tactic': 'Execution',
    'query_table': 'DeviceEvents',
    'query_filter': {'DeviceName': 'vm-db-01', 'FileName': 'mimikatz.exe'},
    'threshold': 1,
    'window_minutes': 120,
    'description': 'Only known-bad binaries. Combine with allowlist watchlist in real Sentinel.',
})
r = httpx.post(f'{SIEM}/rules/evaluate').json()
tuned = [a for a in r['alerts_created'] if a['rule'] == 'DB-server new process (tuned)']
print(f'Tuned rule fired {len(tuned)} alert(s) - just the real threat.')

# Clean up the over-broad rule so it stops producing noise
over_broad = next((x for x in httpx.get(f'{SIEM}/rules').json() if x['name']=='DB-server new process (over-broad)'), None)
if over_broad:
    httpx.delete(f'{SIEM}/rules/{over_broad["id"]}')
    print(f'Deleted over-broad rule {over_broad["id"]}.')

## 2.3 MITRE ATT&CK coverage

Every rule should map to a MITRE ATT&CK tactic so you can see where in the adversary lifecycle you have visibility.

```
Reconnaissance -> Resource Development -> Initial Access -> Execution -> Persistence ->
Privilege Escalation -> Defense Evasion -> Credential Access -> Discovery ->
Lateral Movement -> Collection -> Command & Control -> Exfiltration -> Impact
```

In [ ]:
MITRE_TACTICS = [
    'Reconnaissance','ResourceDevelopment','InitialAccess','Execution',
    'Persistence','PrivilegeEscalation','DefenseEvasion','CredentialAccess',
    'Discovery','LateralMovement','Collection','CommandAndControl',
    'Exfiltration','Impact',
]
rules = httpx.get(f'{SIEM}/rules').json()
covered = {r['tactic'] for r in rules if r['tactic']}

print('=== MITRE ATT&CK Coverage ===\n')
for tactic in MITRE_TACTICS:
    matching = [r['name'] for r in rules if r['tactic'] == tactic]
    if matching:
        print(f'  ✅ {tactic}')
        for name in matching:
            print(f'       - {name}')
    else:
        print(f'  ⬜ {tactic} - no detection')
pct = len(covered)/len(MITRE_TACTICS)*100
print(f'\nCoverage: {len(covered)}/{len(MITRE_TACTICS)} tactics ({pct:.0f}%)')
print('\n💡 Aim for at least one detection in InitialAccess, Execution, and LateralMovement before anything else.')

## 2.4 View generated alerts

Each alert carries:
- **Severity** — business impact
- **Tactic** — adversary-lifecycle stage
- **Entities** — *who* or *what* (user, host, IP)
- **Evidence** — sample events that triggered the detection

In [ ]:
alerts = httpx.get(f'{SIEM}/alerts').json()
print(f'=== All Alerts ({len(alerts)} total) ===\n')
for a in alerts[:10]:
    sev = {'Critical':'🟣','High':'🔴','Medium':'🟡','Low':'🟢'}.get(a['severity'], '⬜')
    entities = json.loads(a['entities']) if isinstance(a['entities'], str) else a['entities']
    print(f'{sev} [{a["status"]}] {a["title"]}')
    if entities: print(f'   Entities: {entities}')
    print(f'   Tactic: {a["tactic"]}  |  Created: {a["created_at"]}\n')

### SC-200 exam: analytics rule fields

| Field | What it controls |
|-------|-----------------|
| **Query** | KQL that identifies the threat |
| **Query frequency** | How often the rule runs (e.g., every 5 min) |
| **Query lookback** | How far back to search (e.g., last 1 hour) |
| **Trigger threshold** | Minimum result count to fire (e.g., > 0) |
| **Entity mapping** | Which fields are accounts, IPs, hosts |
| **MITRE tactics** | ATT&CK classification |
| **Alert grouping** | Group related alerts (by entity, time, ...) |
| **Event grouping** | How many events attach per alert |
| **Suppression** | Silence the rule after it fires (e.g., 1h) |

**Next**: [Notebook 3 — Incidents and Automation](03_incidents_and_automation.ipynb)